In [15]:
DATA_NPZ_PATH = "../datasets/yield_preprocessed.npz"   # adjust if needed

# Sampling on test set
N_TEST_SAMPLES = 50          # number of test instances to explain
SAMPLE_SEED = 123

# Exact coalition enumeration size (per instance)
K_SUBFEATURES = 10           # exact methods run on K features per instance

# Loss: uses log-loss if predict_proba exists, else 0-1 loss
LOSS_NAME = 'mse'  # 'mse' or 'mae'

# Where to save outputs
OUT_DIR = "./outputs_local"

from sklearn.utils.multiclass import type_of_target
TASK = 'classification' if type_of_target(y_train) in ['binary','multiclass','multiclass-multioutput','multilabel-indicator'] else 'regression'
all_labels = np.unique(y_train) if TASK=='classification' else None
LOSS_NAME = None  # None -> default (log_loss for classification, mse for regression)

In [16]:
import os, sys
import numpy as np
import pandas as pd

from itertools import combinations
from typing import List, Sequence, Dict, Any, Tuple

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

os.makedirs(OUT_DIR, exist_ok=True)

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model

from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

print("Repo root:", repo_root)

Repo root: 


In [17]:
def load_yield_npz(npz_path: str):
    """Loads `yield_preprocessed.npz` robustly.

    Expected keys (any of these patterns are handled):
    - X_train, y_train, X_test, y_test
    - X, y  (will split)
    - data, target (will split)
    """
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"Could not find npz at: {npz_path}\nCurrent working dir: {os.getcwd()}")
    d = np.load(npz_path, allow_pickle=True)

    keys = list(d.keys())
    print("NPZ keys:", keys)

    if all(k in d for k in ["X_train", "y_train", "X_test", "y_test"]):
        X_train, y_train, X_test, y_test = d["X_train"], d["y_train"], d["X_test"], d["y_test"]
        return X_train, y_train, X_test, y_test

    if all(k in d for k in ["X", "y"]):
        X, y = d["X"], d["y"]
        return train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if len(np.unique(y)) < len(y) else None)

    if all(k in d for k in ["data", "target"]):
        X, y = d["data"], d["target"]
        return train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if len(np.unique(y)) < len(y) else None)

    # last resort: try to infer arrays
    arrs = [d[k] for k in keys]
    raise ValueError(f"Unrecognized npz structure. Keys={keys}. Please adapt loader.")

In [18]:
X_train, X_test, y_train, y_test = load_yield_npz(DATA_NPZ_PATH)

print("Shapes:")
print("  X_train:", X_train.shape)
print("  y_train:", y_train.shape)
print("  X_test :", X_test.shape)
print("  y_test :", y_test.shape)

# Ensure 2D X and 1D y
X_train = np.asarray(X_train)
X_test  = np.asarray(X_test)
y_train = np.asarray(y_train).ravel()
y_test  = np.asarray(y_test).ravel()

n_features = X_train.shape[1]
print("n_features:", n_features)

NPZ keys: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names']
Shapes:
  X_train: (800, 4)
  y_train: (200, 4)
  X_test : (800,)
  y_test : (200,)
n_features: 4


In [11]:
MODEL_NAMES = ["logistic", "svm", "random_forest"]

trained_models = {}
for m in MODEL_NAMES:
    model = get_model(m, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m] = model

print("Trained:", list(trained_models.keys()))

Trained: ['logistic', 'svm', 'random_forest']


In [19]:
# If y is continuous -> train regression variants.

from sklearn.utils.multiclass import type_of_target

y_type = type_of_target(y_train)
print("Detected target type:", y_type)

trained_models = {}

if y_type in ["binary", "multiclass", "multiclass-multioutput", "multilabel-indicator"]:
    MODEL_NAMES = ["logistic", "svm", "random_forest"]
    for m in MODEL_NAMES:
        model = get_model(m, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        trained_models[m] = model
else:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.svm import SVR
    from sklearn.ensemble import RandomForestRegressor

    MODEL_NAMES = ["logistic", "svm", "random_forest"]  # keep names for reporting

    def get_reg_model(name: str, random_state: int = 42):
        name = name.lower()
        if name == "logistic":
            return Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(random_state=random_state)),
            ])
        if name == "svm":
            return Pipeline([
                ("scaler", StandardScaler()),
                ("svr", SVR(kernel="rbf")),
            ])
        if name == "random_forest":
            return RandomForestRegressor(
                n_estimators=300,
                random_state=random_state,
                n_jobs=-1
            )
        raise ValueError(f"Unknown model: {name}")

    for m in MODEL_NAMES:
        model = get_reg_model(m, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        trained_models[m] = model

print("Trained:", list(trained_models.keys()))

Detected target type: continuous
Trained: ['logistic', 'svm', 'random_forest']


In [29]:
# Compatible with gt_concepts APIs: exposes `data.indices`, `data.data_names` and is callable.

from dataclasses import dataclass
from typing import List, Iterator

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]

    def __len__(self) -> int:
        return len(self.indices)

    def __iter__(self) -> Iterator[int]:
        return iter(self.indices)

    def __getitem__(self, i: int) -> int:
        return self.indices[i]

class LocalLossUtility:
    def __init__(
        self,
        model,
        x_full: np.ndarray,
        y_true,
        baseline_full: np.ndarray,
        feature_subset: np.ndarray,
        task: str,
        all_labels=None,
        loss: str | None = None,
    ):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.K = int(len(self.feature_subset))

        self.task = task  # "classification" or "regression"
        self.y_true = y_true
        self.all_labels = all_labels

        # default losses
        if loss is None:
            loss = "log_loss" if self.task == "classification" else "mse"
        self.loss = loss.lower()

        self.data = FeaturePlayers(
        indices=list(range(len(feature_subset))),
        data_names=[f"f{j}" for j in feature_subset]
)

    def _x_for_coalition(self, S):
    # baseline as 1D
        x = self.baseline_full.copy()

        for local_j in S:
            global_j = int(self.feature_subset[int(local_j)])
            x[global_j] = self.x_full[global_j]

        return x.reshape(1, -1)  # sklearn expects 2D

    def _loss_value(self, X_in) -> float:
        if self.task == "classification":
            proba = self.model.predict_proba(X_in)[0]
            # log_loss expects [y_true] and [proba] aligned with labels
            return float(log_loss([self.y_true], [proba], labels=self.all_labels))
        else:
            y_pred = float(np.asarray(self.model.predict(X_in)).ravel()[0])
            if self.loss == "mse":
                return float((y_pred - float(self.y_true)) ** 2)
            if self.loss == "mae":
                return float(abs(y_pred - float(self.y_true)))
            raise ValueError(f"Unknown regression loss: {self.loss}")

    def __call__(self, S):
        X_in = self._x_for_coalition(S)
        return -self._loss_value(X_in)

In [31]:
rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

records = []

for model_name, model in trained_models.items():
    for idx in test_idx:
        print("X_test type:", type(X_test), "dtype:", getattr(X_test, "dtype", None), "shape:", getattr(X_test, "shape", None))
        tmp = X_test[test_idx[0]]
        print("one item type:", type(tmp), "dtype:", getattr(tmp, "dtype", None), "shape:", getattr(tmp, "shape", None))
        if hasattr(tmp, "dtype") and tmp.dtype == object:
            print("object item inner type:", type(tmp.ravel()[0]))
        x = X_test[idx]
        y = y_test[idx]

        feat_subset = pick_topk_features_by_deviation(x, baseline_full, K_SUBFEATURES)
        util = LocalLossUtility(
            model=model,
            x_full=x,
            y_true=y,
            baseline_full=baseline_full,
            feature_subset=feat_subset,
            task=TASK,
            all_labels=all_labels,
            loss=LOSS_NAME,
        )

        for method_name, method_fn in METHODS:
            # Each method returns an allocation over players 0..K-1
            res = method_fn(util)
            alloc = getattr(res, 'values', res)
            alloc = np.asarray(alloc, dtype=float).ravel()

            # Map back to full feature space (others left as 0)
            full_alloc = np.zeros(n_features, dtype=float)
            for local_j, global_j in enumerate(feat_subset):
                full_alloc[global_j] = alloc[local_j]

            rec = {
                "model": model_name,
                "test_index": int(idx),
                "y_true": y,
                "method": method_name,
                "k": int(len(feat_subset)),
                "feature_subset": feat_subset.tolist(),
                "alloc_full": full_alloc,
                "alloc_subset": alloc,
            }
            records.append(rec)

df = pd.DataFrame(records)
df.head()

X_test type: <class 'numpy.ndarray'> dtype: int64 shape: (800,)
one item type: <class 'numpy.int64'> dtype: int64 shape: ()


In [32]:
import numpy as np

path = "../datasets/yield_preprocessed.npz"
data = np.load(path, allow_pickle=True)

print("Keys:", data.files)
for k in data.files:
    arr = data[k]
    print(k, "shape:", arr.shape, "dtype:", arr.dtype, "min/max:", (arr.min(), arr.max()) if arr.size and np.issubdtype(arr.dtype, np.number) else "NA")

Keys: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names']
X_train shape: (800, 4) dtype: float64 min/max: (np.float64(84.76890922685757), np.float64(5576.028367256043))
X_test shape: (200, 4) dtype: float64 min/max: (np.float64(88.29956974450485), np.float64(5621.546484797367))
y_train shape: (800,) dtype: int64 min/max: (np.int64(0), np.int64(2))
y_test shape: (200,) dtype: int64 min/max: (np.int64(0), np.int64(2))
feature_names shape: (4,) dtype: <U13 min/max: NA


In [33]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.utils.multiclass import type_of_target
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../datasets/yield_preprocessed.npz"
OUT_DIR = "./outputs_local"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

# Since N_features is small (4), we usually don't need to limit subfeatures.
# We will use min(N_features, K_SUBFEATURES) later.
K_SUBFEATURES = 10

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

def load_and_fix_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    d = np.load(npz_path, allow_pickle=True)

    # Extract raw arrays
    X_train = d["X_train"] if "X_train" in d else d.get("X", None)
    y_train = d["y_train"] if "y_train" in d else d.get("y", None)
    X_test  = d["X_test"]  if "X_test" in d  else None
    y_test  = d["y_test"]  if "y_test" in d  else None

    # If X_test is (800,) and we have 4 features, it should likely be (200, 4)
    if X_test is not None and X_train is not None:
        if X_test.ndim == 1 and X_train.ndim == 2:
            n_feats = X_train.shape[1]
            print(f"Reshaping X_test from {X_test.shape} to (-1, {n_feats})...")
            X_test = X_test.reshape(-1, n_feats)

    # If X_train is (800, 4) but y_train is (200, 4), we assume y_train needs to match X_train rows
    if X_train.shape[0] != y_train.shape[0]:
        print(f"Warning: X_train rows {X_train.shape[0]} != y_train rows {y_train.shape[0]}.")
        # Assuming standard split logic, we proceed, but this might fail .fit()
        # For this script, we will ensure they match or slice X to match y.
        min_len = min(X_train.shape[0], y_train.shape[0])
        X_train = X_train[:min_len]
        y_train = y_train[:min_len]

    return X_train, y_train, X_test, y_test

# Load Data
X_train, y_train, X_test, y_test = load_and_fix_data(DATA_NPZ_PATH)

# Standardize formats
X_train = np.asarray(X_train, dtype=float)
X_test  = np.asarray(X_test, dtype=float)
y_train = np.asarray(y_train).ravel() # Ensure 1D targets
y_test  = np.asarray(y_test).ravel()

print("Final Shapes:")
print("  X_train:", X_train.shape)
print("  y_train:", y_train.shape)
print("  X_test :", X_test.shape)
print("  y_test :", y_test.shape)

n_features = X_train.shape[1]

# Now that data is loaded, we can define the task
if type_of_target(y_train) in ['binary', 'multiclass', 'multiclass-multioutput', 'multilabel-indicator']:
    TASK = 'classification'
    LOSS_NAME = 'log_loss'
    all_labels = np.unique(y_train)
else:
    TASK = 'regression'
    LOSS_NAME = 'mse'
    all_labels = None

print(f"Task detected: {TASK}, Loss: {LOSS_NAME}")

# Define Baseline (Reference)
# Common practice: Mean of training data
baseline_full = np.mean(X_train, axis=0)
print("Baseline (mean) shape:", baseline_full.shape)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

print("Models trained:", list(trained_models.keys()))

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, task, all_labels=None, loss='mse'):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.task = task
        self.y_true = y_true
        self.all_labels = all_labels
        self.loss = loss

        # Setup GT player interface
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[f"f{j}" for j in feature_subset]
        )

    def _x_for_coalition(self, S):
        # Start with baseline
        x = self.baseline_full.copy()

        # Fill in active features
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]

        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        if self.task == "classification":
            if hasattr(self.model, "predict_proba"):
                proba = self.model.predict_proba(X_in)
                # log_loss requires (n_samples, n_classes) or (n_samples,)
                return float(log_loss([self.y_true], proba, labels=self.all_labels))
            else:
                # Fallback for models without probability (rare in standard sklearn classifiers)
                pred = self.model.predict(X_in)[0]
                return 0.0 if pred == self.y_true else 1.0
        else:
            y_pred = float(self.model.predict(X_in).ravel()[0])
            if self.loss == "mse":
                return float((y_pred - float(self.y_true)) ** 2)
            if self.loss == "mae":
                return float(abs(y_pred - float(self.y_true)))
            return float((y_pred - float(self.y_true)) ** 2)

    def __call__(self, S):
        X_in = self._x_for_coalition(S)
        return -self._loss_value(X_in)

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

records = []

print(f"Running explanation on {len(test_idx)} samples...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        x = X_test[idx] # This is now safe because X_test is (N, 4)
        y = y_test[idx]

        # Since n_features (4) < K (10), we always use ALL features.
        # No need for top-k selection.
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model,
            x_full=x,
            y_true=y,
            baseline_full=baseline_full,
            feature_subset=feat_subset,
            task=TASK,
            all_labels=all_labels,
            loss=LOSS_NAME,
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                # Handle return formats (some return objects, some arrays)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                rec = {
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": y,
                    "method": method_name,
                    "alloc_full": alloc.tolist() # already full size since we didn't subset
                }
                records.append(rec)
            except Exception as e:
                print(f"Error in {method_name} for idx {idx}: {e}")

df = pd.DataFrame(records)
print("Experiment Complete. First 5 rows:")
print(df.head())

# Save
df.to_csv(os.path.join(OUT_DIR, "loss_utility_results.csv"), index=False)

Final Shapes:
  X_train: (800, 4)
  y_train: (800,)
  X_test : (200, 4)
  y_test : (200,)
Task detected: classification, Loss: log_loss
Baseline (mean) shape: (4,)
Models trained: ['logistic', 'svm', 'random_forest']
Running explanation on 50 samples...
Experiment Complete. First 5 rows:
      model  test_index  y_true          method  \
0  logistic          34       0         shapley   
1  logistic          34       0      least_core   
2  logistic          34       0       nucleolus   
3  logistic          34       0          gately   
4  logistic          34       0  shapley_shubik   

                                          alloc_full  
0  [0.0016468224775299893, 1.1416009598226429, 0....  
1  [-0.7083753610753626, 0.42003270881648813, -0....  
2  [-0.6876510178655387, 0.39985471053328764, -0....  
3  [-0.7006629951469927, 0.4125426996861481, -0.2...  
4                           [0.25, 0.25, 0.25, 0.25]  


In [34]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../datasets/yield_preprocessed.npz"
OUT_DIR = "./outputs_local"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import existing implementations
from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    # 1. Load exactly the four arrays (fail if keys missing)
    try:
        X_train = d["X_train"].astype(float)
        X_test  = d["X_test"].astype(float)
        y_train = d["y_train"].astype(int).ravel() # Ensure 1D
        y_test  = d["y_test"].astype(int).ravel()
    except KeyError as e:
        raise KeyError(f"Missing expected key in NPZ: {e}. Keys found: {list(d.keys())}")

    # 2. Load feature names if available, else default
    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    # 3. Strict Shape Assertions (Fail Fast)
    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")

    assert X_train.ndim == 2, "X_train must be 2D"
    assert X_test.ndim == 2,  "X_test must be 2D"
    assert X_train.shape[0] == y_train.shape[0], f"Mismatch: X_train rows {X_train.shape[0]} != y_train {y_train.shape[0]}"
    # But for a standard ML setup, they must match.
    assert X_test.shape[0] == y_test.shape[0], f"Mismatch: X_test rows {X_test.shape[0]} != y_test {y_test.shape[0]}"
    assert len(feature_names) == X_train.shape[1], "Mismatch between feature_names length and X columns"

    return X_train, y_train, X_test, y_test, feature_names

# Load Data
X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]

# Ensure labels are a sorted list for consistent log_loss usage
all_labels = sorted(np.unique(y_train).tolist())
print(f"Labels detected: {all_labels}")

# Baseline: Mean of training data
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)

    # Safety check: Ensure it is a classifier
    if not is_classifier(model):
         # If get_model returns a regressor but data implies classification, warn/error
         print(f"Warning: {m_name} does not seem to be a classifier (is_classifier=False).")

    model.fit(X_train, y_train)
    trained_models[m_name] = model

print("Models trained:", list(trained_models.keys()))

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true) # Ensure int
        self.all_labels = all_labels

        # Setup GT player interface with REAL feature names
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[feature_names[j] for j in feature_subset]
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        # Check if model supports probability
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            # Fallback for models like SVM (if probability=False) or others
            # Warning: log_loss is not strictly defined without probs,
            # but we can use 0-1 loss or raise error.
            # Here we try to interpret decision function or hard predict.
            pred = self.model.predict(X_in)[0]
            # Simple 0-1 Loss fallback
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        X_in = self._x_for_coalition(S)
        return -self._loss_value(X_in)

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

records = []

print(f"\nRunning explanation on {len(test_idx)} samples using {len(METHODS)} methods...")

for model_name, model in trained_models.items():
    # Check if this model might fail log_loss (e.g. SVM without probability=True)
    if model_name == "svm" and not getattr(model, "probability", True) and not hasattr(model, "predict_proba"):
        print(f"Warning: SVM model might lack predict_proba. Fallback 0-1 loss will be used.")

    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]

        # Use ALL features (exact computation)
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model,
            x_full=x,
            y_true=y,
            baseline_full=baseline_full,
            feature_subset=feat_subset,
            feature_names=feature_names,
            all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                # Create named dictionary for clarity
                alloc_named = {feature_names[j]: float(alloc[j]) for j in range(n_features)}

                rec = {
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist(),
                    "alloc_named": alloc_named
                }
                records.append(rec)
            except Exception as e:
                print(f"Error in {method_name} (Model: {model_name}, Idx: {idx}): {e}")

df = pd.DataFrame(records)
print("\nExperiment Complete. First 5 rows:")
print(df.head())

out_file = os.path.join(OUT_DIR, "loss_utility_results.csv")
df.to_csv(out_file, index=False)
print(f"\nResults saved to: {out_file}")

Loading data from ../datasets/yield_preprocessed.npz...
Shapes -> X_train: (800, 4), y_train: (800,), X_test: (200, 4)
Labels detected: [0, 1, 2]

Training models...
Models trained: ['logistic', 'svm', 'random_forest']

Running explanation on 50 samples using 6 methods...

Experiment Complete. First 5 rows:
      model  test_index  y_true          method  \
0  logistic          34       0         shapley   
1  logistic          34       0      least_core   
2  logistic          34       0       nucleolus   
3  logistic          34       0          gately   
4  logistic          34       0  shapley_shubik   

                                          alloc_full  \
0  [0.0016468224775299893, 1.1416009598226429, 0....   
1  [-0.7083753610753626, 0.42003270881648813, -0....   
2  [-0.6876510178655387, 0.39985471053328764, -0....   
3  [-0.7006629951469927, 0.4125426996861481, -0.2...   
4                           [0.25, 0.25, 0.25, 0.25]   

                                         alloc_

In [53]:
import os
import sys
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List, Callable, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(".."))

# Import GT methods
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../datasets/yield_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_local/loss_utility_results.csv"
OUT_DIR = Path("./output/quantus_evaluation_soil")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()
    return X_train, X_test, y_train, y_test

# 2. robust perturbation & distance functions

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    """
    Safely perturbs tabular data by replacing with baseline.
    """
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))


def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,  # Faithfulness uses subset_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,   # Selectivity uses patch_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1, # Monotonicity uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1, # PixelFlipping uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=5, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    for k in top_k_list:
        if k > X_test_2d.shape[1]: continue
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k)

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "shapley_shubik": exact_shapley_shubik,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=y,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    X_train, X_test, y_train, y_test = load_data_strict()
    baseline = np.mean(X_train, axis=0)

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    # Quantus Input: (N, 1, F)
    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    all_eval_rows = []
    metrics = get_metrics(baseline)

    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        all_labels = sorted(np.unique(y_train).tolist())

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    if isinstance(val_str, str):
                        val_arr = np.array(json.loads(val_str))
                    else:
                        val_arr = np.array(val_str)
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            # Global Metrics
            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            # Aopc
            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            # Robustness
            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantus_results_full_extended.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model          method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic         shapley           0.800064               0.845704     0.225566          0.62        0.225566    0.414236    1.024434     0.468445                  0.007081         0.011202
1        logistic      least_core           0.893539               0.893002     0.228585          0.46        0.228585    0.312303    1.191842     0.349226                  0.009040         0.015021
2        logistic       nucleolus           0.904980               0.897384     0.225685          0.46        0.225685    0.289515    1.213172     0.348390                  0.010809         0.016198
3        logistic          gately           0.865701               0.857211     0.226936          0.46        0.226936    0.299502    1.203935     0.351188                  0.007057         0.015655
4    

In [ ]:
import os
import ast
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import log_loss

# Config
DATA_NPZ_PATH = "../datasets/yield_preprocessed.npz"
RESULTS_CSV = "./outputs_local/loss_utility_results.csv"
OUTPUT_DIR = "./outputs_local/insertion_deletion_plots"

RANDOM_STATE = 42
N_STEPS = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)

METHOD_NAME_MAP = {
    "shapley": "Shapley",
    "banzhaf": "Banzhaf",
    "gately": "Gately",
    "least_core": "Least Core",
    "nucleolus": "Nucleolus"
}

# Import project modules
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from models.train_model import get_model

# Load data
data = np.load(DATA_NPZ_PATH, allow_pickle=True)
X_train = data["X_train"].astype(float)
X_test  = data["X_test"].astype(float)
y_train = data["y_train"].astype(int).ravel()
y_test  = data["y_test"].astype(int).ravel()

if "feature_names" in data:
    feature_names = data["feature_names"].tolist()
else:
    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

baseline_full = np.mean(X_train, axis=0)
all_labels = sorted(np.unique(y_train).tolist())

# Train models
MODEL_NAMES = sorted(pd.read_csv(RESULTS_CSV)["model"].unique().tolist())

trained_models = {}
for model_name in MODEL_NAMES:
    print(f"Training model: {model_name}")
    model = get_model(model_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

# Load attributions
df = pd.read_csv(RESULTS_CSV)

def parse_alloc(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)
    return np.array(x, dtype=float)

df["alloc_full_parsed"] = df["alloc_full"].apply(parse_alloc)

# Loss function
def compute_instance_loss(model, x_row, y_true):
    x_row = x_row.reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x_row)
        return float(log_loss([y_true], proba, labels=all_labels))
    else:
        pred = model.predict(x_row)[0]
        return 0.0 if pred == y_true else 1.0

# Curves
def deletion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = x.copy()
        x_mod[order[:k]] = baseline[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def insertion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = baseline.copy()
        x_mod[order[:k]] = x[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def curve_auc(x, y):
    return float(np.trapezoid(y, x))

# Main loop
summary_rows = []

for model_name in sorted(df["model"].unique()):
    print(f"\nProcessing model: {model_name}")
    model = trained_models[model_name]

    methods = sorted([m for m in df[df["model"] == model_name]["method"].unique()
    if m != "shapley_shubik"
])

    deletion_plot_data = {}
    insertion_plot_data = {}

    for method_name in methods:
        pretty_name = METHOD_NAME_MAP.get(method_name, method_name)

        sub = df[(df["model"] == model_name) & (df["method"] == method_name)]

        deletion_curves = []
        insertion_curves = []

        for _, row in sub.iterrows():
            idx = int(row["test_index"])
            y_true = int(row["y_true"])
            attribution = row["alloc_full_parsed"]

            x = X_test[idx]

            x_del, y_del = deletion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)
            x_ins, y_ins = insertion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)

            deletion_curves.append(y_del)
            insertion_curves.append(y_ins)

        deletion_curves = np.array(deletion_curves)
        insertion_curves = np.array(insertion_curves)

        del_mean = deletion_curves.mean(axis=0)
        del_std = deletion_curves.std(axis=0)

        # Normalize
        del_mean = (del_mean - del_mean[0]) / (del_mean[-1] - del_mean[0] + 1e-8)
        del_std = del_std / (del_mean[-1] - del_mean[0] + 1e-8)

        ins_mean = insertion_curves.mean(axis=0)
        ins_std = insertion_curves.std(axis=0)

        # Normalize
        ins_mean = (ins_mean - ins_mean[-1]) / (ins_mean[0] - ins_mean[-1] + 1e-8)
        ins_std = ins_std / (ins_mean[0] - ins_mean[-1] + 1e-8)

        del_auc = curve_auc(x_del, del_mean)
        ins_auc = curve_auc(x_ins, ins_mean)

        summary_rows.append({
            "model": model_name,
            "method": method_name,
            "deletion_auc_loss": del_auc,
            "insertion_auc_loss": ins_auc
        })

        deletion_plot_data[pretty_name] = (x_del, del_mean, del_std, del_auc)
        insertion_plot_data[pretty_name] = (x_ins, ins_mean, ins_std, ins_auc)

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in deletion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Perturbed")
    plt.ylabel("Loss")
    plt.title(f"Deletion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/deletion_{model_name}.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in insertion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Restored")
    plt.ylabel("Loss")
    plt.title(f"Insertion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/insertion_{model_name}.png", dpi=300)
    plt.close()

# Save summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/auc_summary.csv", index=False)

print("\nDone!")
print(summary_df)

Training model: logistic
Training model: random_forest
Training model: svm

Processing model: logistic

Processing model: random_forest

Processing model: svm

Done!
            model      method  deletion_auc_loss  insertion_auc_loss
0        logistic     banzhaf           0.856106            0.186611
1        logistic      gately           0.617490            0.296175
2        logistic  least_core           0.616096            0.311025
3        logistic   nucleolus           0.609584            0.302803
4        logistic     shapley           0.850285            0.176244
5   random_forest     banzhaf           1.176392            0.165546
6   random_forest      gately           0.664640            0.723229
7   random_forest  least_core           0.758000            0.447167
8   random_forest   nucleolus           0.797414            0.445692
9   random_forest     shapley           0.865663            0.732175
10            svm     banzhaf           0.781651            0.178335
11    

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv("./outputs_local/loss_utility_results.csv")
df["alloc_full"] = df["alloc_full"].apply(lambda x: np.array(eval(x)))

features = ["Irrigation", "Nitrogen", "Precipitation", "Herbicide"]

GT = np.array([3.7065, 2.7649, 2.0523, 0.8179])

GT = GT / (np.linalg.norm(GT) + 1e-8)

results = []

for (model, method), group in df.groupby(["model", "method"]):

    # Stack all local attributions
    attribs = np.vstack(group["alloc_full"].values)

    global_attr = np.mean(np.abs(attribs), axis=0)

    # Normalize
    global_attr = global_attr / (np.linalg.norm(global_attr) + 1e-8)


    spearman = spearmanr(GT, global_attr).correlation

    # Cosine
    cosine = cosine_similarity(GT.reshape(1, -1),
                               global_attr.reshape(1, -1))[0][0]

    # Errors
    mae = mean_absolute_error(GT, global_attr)
    rmse = np.sqrt(mean_squared_error(GT, global_attr))

    results.append({
        "model": model,
        "method": method,
        "spearman": spearman,
        "cosine": cosine,
        "mae": mae,
        "rmse": rmse
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(["model", "spearman"], ascending=[True, False])

print("\n===== Yield Dataset: Global Comparison =====\n")
print(results_df.round(4).to_string(index=False))


===== Yield Dataset: Global Comparison =====

        model         method  spearman  cosine    mae   rmse
     logistic        banzhaf       0.4  0.9041 0.1937 0.2190
     logistic         gately       0.4  0.9149 0.1783 0.2063
     logistic     least_core       0.4  0.9242 0.1691 0.1946
     logistic      nucleolus       0.4  0.9142 0.1771 0.2071
     logistic        shapley       0.4  0.8869 0.1889 0.2379
     logistic shapley_shubik       0.4  0.9097 0.1891 0.2125
random_forest        shapley       0.8  0.9455 0.1385 0.1650
random_forest        banzhaf       0.6  0.9160 0.2034 0.2049
random_forest         gately       0.4  0.8899 0.2018 0.2347
random_forest     least_core       0.4  0.8812 0.2141 0.2437
random_forest      nucleolus       0.4  0.8839 0.2123 0.2410
random_forest shapley_shubik       0.4  0.9160 0.1850 0.2049
          svm        banzhaf       0.4  0.9078 0.1732 0.2147
          svm         gately       0.4  0.9094 0.1856 0.2128
          svm     least_core       0.4

In [4]:
import ast
import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

# Config
RESULTS_CSV = "./outputs_local/loss_utility_results.csv"

# Make sure this order matches the attribution vector order in alloc_full
GROUND_TRUTH_FEATURES = ["Irrigation", "Nitrogen", "Precipitation", "Herbicide"]
GROUND_TRUTH_VALUES = np.array([3.7065, 2.7649, 2.0523, 0.8179], dtype=float)

EXCLUDE_METHODS = {"shapley_shubik"}

# Load results
df = pd.read_csv(RESULTS_CSV)

def parse_alloc(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)
    return np.array(x, dtype=float)

df["alloc_full_parsed"] = df["alloc_full"].apply(parse_alloc)

if EXCLUDE_METHODS:
    df = df[~df["method"].isin(EXCLUDE_METHODS)].copy()

# Ground truth
GT = GROUND_TRUTH_VALUES.copy()

# Spearman is rank-based, so no normalization is needed there.
# For cosine / MAE / RMSE, use L1 normalization so the vectors are
# comparable as global importance distributions.
GT_L1 = GT / (GT.sum() + 1e-8)

# Metrics
def compute_global_metrics(gt_raw: np.ndarray, gt_l1: np.ndarray, attribs: np.ndarray):
    """
    gt_raw: raw global ground truth vector, shape (p,)
    gt_l1: L1-normalized ground truth vector, shape (p,)
    attribs: local attributions stacked across samples, shape (n_samples, p)
    """

    # Global attribution from local explanations
    global_attr_abs = np.mean(np.abs(attribs), axis=0)

    # L1 normalize for cosine / MAE / RMSE
    global_attr_l1 = global_attr_abs / (global_attr_abs.sum() + 1e-8)

    # Spearman on raw vectors (scale-invariant)
    spearman_val = spearmanr(gt_raw, global_attr_abs).correlation

    # Cosine on L1-normalized vectors
    cosine_val = cosine_similarity(
        gt_l1.reshape(1, -1),
        global_attr_l1.reshape(1, -1)
    )[0][0]

    # MAE / RMSE on L1-normalized vectors
    mae_val = mean_absolute_error(gt_l1, global_attr_l1)
    rmse_val = np.sqrt(mean_squared_error(gt_l1, global_attr_l1))

    return {
        "spearman": float(spearman_val),
        "cosine": float(cosine_val),
        "mae": float(mae_val),
        "rmse": float(rmse_val),
        "gt_raw": gt_raw,
        "gt_l1": gt_l1,
        "global_attr_abs": global_attr_abs,
        "global_attr_l1": global_attr_l1,
    }

# Main evaluation
results = []

for (model, method), group in df.groupby(["model", "method"]):
    attribs = np.vstack(group["alloc_full_parsed"].values)

    # Safety check
    if attribs.shape[1] != len(GT):
        raise ValueError(
            f"Feature dimension mismatch for model={model}, method={method}: "
            f"attribs has {attribs.shape[1]} features but GT has {len(GT)}"
        )

    metrics = compute_global_metrics(GT, GT_L1, attribs)

    results.append({
        "model": model,
        "method": method,
        "spearman": metrics["spearman"],
        "cosine": metrics["cosine"],
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["model", "spearman"], ascending=[True, False]).reset_index(drop=True)

print("\n===== Yield Dataset: Global Ground Truth Comparison =====\n")
print(results_df.round(4).to_string(index=False))

# Optional: show global attribution vectors too
print("\n===== Ground Truth Feature Order =====")
print(GROUND_TRUTH_FEATURES)
print("Ground truth (raw):", np.round(GT, 4))
print("Ground truth (L1-normalized):", np.round(GT_L1, 4))

print("\n===== Per-model/method Global Attributions =====\n")
for (model, method), group in df.groupby(["model", "method"]):
    attribs = np.vstack(group["alloc_full_parsed"].values)
    global_attr_abs = np.mean(np.abs(attribs), axis=0)
    global_attr_l1 = global_attr_abs / (global_attr_abs.sum() + 1e-8)

    row = {
        "model": model,
        "method": method,
    }
    for feat, val in zip(GROUND_TRUTH_FEATURES, global_attr_abs):
        row[f"{feat}_abs_mean"] = val
    for feat, val in zip(GROUND_TRUTH_FEATURES, global_attr_l1):
        row[f"{feat}_l1_norm"] = val

    print(pd.Series(row).to_string())
    print("-" * 80)

# Optional: save results
OUT_CSV = "./outputs_local/global_ground_truth_comparison.csv"
results_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved summary to: {OUT_CSV}")


===== Yield Dataset: Global Ground Truth Comparison =====

        model     method  spearman  cosine    mae   rmse
     logistic    banzhaf       0.4  0.9041 0.1125 0.1243
     logistic     gately       0.4  0.9149 0.0930 0.1117
     logistic least_core       0.4  0.9242 0.0882 0.1054
     logistic  nucleolus       0.4  0.9142 0.0930 0.1123
     logistic    shapley       0.4  0.8869 0.1132 0.1380
random_forest    shapley       0.8  0.9455 0.0780 0.0919
random_forest    banzhaf       0.6  0.9160 0.1105 0.1114
random_forest     gately       0.4  0.8899 0.1089 0.1281
random_forest least_core       0.4  0.8812 0.1138 0.1324
random_forest  nucleolus       0.4  0.8839 0.1129 0.1309
          svm    banzhaf       0.4  0.9078 0.0959 0.1182
          svm     gately       0.4  0.9094 0.0961 0.1150
          svm least_core       0.4  0.9181 0.0921 0.1094
          svm  nucleolus       0.4  0.9101 0.0958 0.1147
          svm    shapley       0.4  0.8832 0.1129 0.1400

===== Ground Truth Feature 